In [2]:
# read global_map
import pickle
global_keys = pickle.load(open("/mnt/shared-storage-gpfs2/beam-gpfs02/yulang/master/nemo_cellflow/preprocessing/pbmc/group/global_keys.pkl", "rb"))
donor_keys, cytokine_keys, cell_type_keys = global_keys['donor'], global_keys['cytokine'], global_keys['cell_type']
from itertools import product
cartesian_keys = list(product(donor_keys, cell_type_keys, cytokine_keys))
group_dict = {k: [] for k in cartesian_keys}

# 三种条件的所有情况做笛卡尔积
len(cartesian_keys)
# read h5ad


19656

In [3]:
cartesian_keys[:10]

[('Donor1', 'B Intermediate/Memory', '4-1BBL'),
 ('Donor1', 'B Intermediate/Memory', 'ADSF'),
 ('Donor1', 'B Intermediate/Memory', 'APRIL'),
 ('Donor1', 'B Intermediate/Memory', 'BAFF'),
 ('Donor1', 'B Intermediate/Memory', 'C3a'),
 ('Donor1', 'B Intermediate/Memory', 'C5a'),
 ('Donor1', 'B Intermediate/Memory', 'CD27L'),
 ('Donor1', 'B Intermediate/Memory', 'CD30L'),
 ('Donor1', 'B Intermediate/Memory', 'CD40L'),
 ('Donor1', 'B Intermediate/Memory', 'CT-1')]

In [5]:
# 按cartessian_key划分细胞
import numpy as np
from tqdm import tqdm
import scipy.sparse as sp
import scanpy as sc
import anndata as ad
pbmc_h5ad_path = '/mnt/shared-storage-gpfs2/beam-gpfs02/yulang/datasets/pbmc/pbmc_parse.h5ad'
adata = ad.read_h5ad(pbmc_h5ad_path)

def iter_adata_rows(
    adata,
    cols=("donor", "cell_type", "cytokine"),
    to_float32: bool = False,
):

    # 1) obs三列一次性取出，避免每行反复做 pandas 索引
    obs_df = adata.obs.loc[:, list(cols)]

    # 2) X：若是稀疏，尽量保证 CSR 以便按行切片更快
    X = adata.X
    if sp.issparse(X) and not sp.isspmatrix_csr(X):
        X = X.tocsr()

    is_sparse = sp.issparse(X)

    for i in tqdm(range(adata.n_obs)):
        row_obs = obs_df.iloc[i]

        # 取obs字段
        out = {"i": i}
        for c in cols:
            out[c] = row_obs[c]

        out["x"] = X[i]
        yield out



processed_cartesian_keys = []
# 用法示例




for item in iter_adata_rows(adata):
    donor, cell_type, cytokine, x = item["donor"], item["cell_type"], item["cytokine"], item["x"]
    group_dict[(donor, cell_type, cytokine)].append(x)
    processed_cartesian_keys.append((donor, cell_type, cytokine))
del adata
    

100%|██████████| 9697974/9697974 [13:43<00:00, 11773.97it/s]


In [6]:
processed_cartesian_keys = list(set(processed_cartesian_keys))
# 按1024 shard_size 进行配对的并保存
control_name = "PBS"
shard_size = 1024
rng = np.random.default_rng(0)
pert_sample_list = []
control_dict = {}

def pad_for_shard_size(array, shard_size):
    N = array.shape[0]
    pad_n = (-N) % shard_size  # number of rows to add
    if pad_n == 0:
        return array, np.empty((0,), dtype=np.int64)
    pad_idx = rng.integers(0, N, size=pad_n)   # with replacement
    array_pad = sp.vstack([array, array[pad_idx]])
    return array_pad, pad_idx

def equal_pert_parts(pert_array, shard_size=1024):
    
    pert_array_pad, _ = pad_for_shard_size(pert_array, shard_size)
    pert_num = pert_array_pad.shape[0]

    
    for sample_idx, i in enumerate(range(0, pert_num, shard_size)):
        pert_sub = pert_array_pad[i:i+shard_size]
        sample = {
            "cartesian_key": pck,
            "cell_matrix": pert_sub,
        }
        pert_sample_list.append(sample)


for pck in tqdm(processed_cartesian_keys):
    if pck[2] == control_name:
        
        control_dict[pck] = sp.vstack(group_dict[pck], format="csr")
    else:
        pert_array = sp.vstack(group_dict[pck], format="csr")
        equal_pert_parts(pert_array, shard_size=shard_size)
    


100%|██████████| 19459/19459 [01:29<00:00, 217.63it/s]


In [7]:
len(pert_sample_list)

23245

In [8]:
with open(f"pert_sample_list.pkl", "wb") as f:
    pickle.dump(pert_sample_list, f)


In [9]:
with open(f"control_dict.pkl", "wb") as f:
    pickle.dump(control_dict, f)